In [ ]:
from sparcl.client import SparclClient
import numpy as np
from astropy.table import Table
from astropy.io import fits

In [ ]:
client = SparclClient()

In [ ]:
dwarf_cat = Table.read("../data/stellar_mass_emline_dwarfs.fits")

## Utilities

In [ ]:
def get_spectra(target_ids: list):
    inc = ['specid', 'redshift', 'flux', 'wavelength', 'spectype', 'specprimary', 'survey', 'program', 'targetid', 'redshift_warning']

    res = client.retrieve_by_specid(specid_list = target_ids, include = inc,
                                    dataset_list = ['DESI-DR1'], limit=1_000_000)


    records = res.records


    array_len = len(records[0].flux)

    
    for record in records:
        assert len(record.flux) == array_len
        assert len(record.wavelength) == array_len


    return records

In [ ]:
def records_to_fits(records, filename, dr_keyword="_dr"):
    """
    Write a list of record dicts to a FITS binary table via astropy.Table.
 
    All records must have 'flux'/'wavelength' arrays of the same length.
    A scalar field that is constant across all records (e.g. '_dr') is
    stored once in the file's header/metadata instead of as a column.
    """
 
    colnames = list(records[0].keys())
    table = Table(rows=records, names=colnames)
    table.write(filename, format="fits", overwrite=True)
    print(f"Wrote {len(records)} records to {filename}")


## retrieve spectrum chunk by chunk

In [ ]:
chunk = 500

In [ ]:
idx0 = chunk

while idx0 < len(dwarf_cat):
    print(idx0)
    target_ids = dwarf_cat["targetid"][idx0:idx0+chunk].tolist()
    records = get_spectra(target_ids)
    print("records retrieved")
    records_to_fits(records, f"../data/spectra/dwarfs_{idx0}_{idx0+chunk-1}.fits")

    idx0 += chunk